- **Group Name**: MVPs
- **Course:** Advanced Deep Learning - AIGC-5500-0NA
- **Professor**: Hossein Pourmodheji
- **Owner:** Nouroz Ali
---


In [15]:
import os
import re
import json
import random
import time
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Experiment settings
DATASET_ID = "SetFit/enron_spam"

SEEDS = [42, 123, 2026]

BATCH_SIZE = 64
EPOCHS = 10
MAX_LENGTH = 256
MAX_VOCAB_SIZE = 10000

EMBEDDING_DIM = 64
HIDDEN_DIM = 64
NUM_CLASSES = 2

LEARNING_RATE = 0.001

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

OUTPUT_DIR = "results/enron_spam_experiment"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Dataset:", DATASET_ID)
print("Device:", DEVICE)
print("Seeds:", SEEDS)

Dataset: SetFit/enron_spam
Device: cpu
Seeds: [42, 123, 2026]


## Load and inspect the dataset

In [16]:
# Download the dataset from Hugging Face
raw_dataset = load_dataset(DATASET_ID)

print("Available splits:", list(raw_dataset.keys()))

for split_name, split_data in raw_dataset.items():
    print(
        split_name,
        "rows:", len(split_data),
        "columns:", split_data.column_names
    )

# Use the official test split when available.
# Otherwise, create a stratified train/test split.
if "train" not in raw_dataset:
    raise ValueError("The dataset does not contain a train split.")

train_df = raw_dataset["train"].to_pandas()

if "test" in raw_dataset:
    test_df = raw_dataset["test"].to_pandas()
else:
    train_df, test_df = train_test_split(
        train_df,
        test_size=0.20,
        random_state=42,
        stratify=train_df["label"]
    )

# Create a validation split from the training data.
train_df, val_df = train_test_split(
    train_df,
    test_size=0.15,
    random_state=42,
    stratify=train_df["label"]
)

print("\nTraining rows:", len(train_df))
print("Validation rows:", len(val_df))
print("Testing rows:", len(test_df))

display(train_df.head())

Repo card metadata block was not found. Setting CardData to empty.


Available splits: ['train', 'test']
train rows: 31716 columns: ['message_id', 'text', 'label', 'label_text', 'subject', 'message', 'date']
test rows: 2000 columns: ['message_id', 'text', 'label', 'label_text', 'subject', 'message', 'date']

Training rows: 26958
Validation rows: 4758
Testing rows: 2000


,message_id,text,label,label_text,subject,message,date
25312,2550,tenaska 11 / 00 and 12 / 00 we have the actual...,0,ham,tenaska 11 / 00 and 12 / 00,we have the actuals for december 2000 . we nee...,2001-02-09
12585,32835,"are youu stil 3 l pa margin - right : 8 "" alig...",1,spam,,"are youu stil 3 l pa margin - right : 8 "" alig...",2005-05-14
16620,4073,"$ 55486 hello ,\ni sent you an email a few day...",1,spam,$ 55486,"hello ,\ni sent you an email a few days ago be...",2004-06-27
6206,28894,etc - event - vince gill & amy grant please se...,0,ham,etc - event - vince gill & amy grant,please see attachment for a christmas to remem...,2001-10-15
22954,25902,"all graphics software available , cheap oem ve...",1,spam,"all graphics software available , cheap oem ve...","good morning ,\nwe we offer latest oem package...",2005-06-28


## Clean the text and prepare labels
This code uses the text column and label column. Labels are expected to be 0 = ham and 1 = spam.

In [17]:
def clean_text(text):
    """Convert email text to lowercase and normalize whitespace."""
    if text is None:
        return ""

    text = str(text).lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def prepare_dataframe(df):
    df = df.copy()

    required_columns = {"text", "label"}
    if not required_columns.issubset(df.columns):
        raise ValueError(
            f"Expected columns {required_columns}, "
            f"but found {list(df.columns)}"
        )

    df = df[["text", "label"]].dropna().copy()
    df["text"] = df["text"].apply(clean_text)
    df["label"] = df["label"].astype(int)

    df = df[df["text"].str.len() > 0]
    df = df[df["label"].isin([0, 1])]

    return df.reset_index(drop=True)


train_df = prepare_dataframe(train_df)
val_df = prepare_dataframe(val_df)
test_df = prepare_dataframe(test_df)

print("Training label counts:")
print(train_df["label"].value_counts().sort_index())

print("\nValidation label counts:")
print(val_df["label"].value_counts().sort_index())

print("\nTest label counts:")
print(test_df["label"].value_counts().sort_index())

print("\nLabel meaning: 0 = ham, 1 = spam")

Training label counts:
label
0    13220
1    13694
Name: count, dtype: int64

Validation label counts:
label
0    2333
1    2418
Name: count, dtype: int64

Test label counts:
label
0     992
1    1008
Name: count, dtype: int64

Label meaning: 0 = ham, 1 = spam


## Build the vocabulary using training data only
The validation and test sets must not be used to build the vocabulary.
PAD_TOKEN = "<PAD>"

In [18]:
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"


def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())


def build_vocabulary(texts, max_vocab_size=MAX_VOCAB_SIZE):
    word_counts = Counter()

    for text in texts:
        word_counts.update(tokenize(text))

    vocabulary = {
        PAD_TOKEN: 0,
        UNK_TOKEN: 1
    }

    for word, _ in word_counts.most_common(max_vocab_size - 2):
        if word not in vocabulary:
            vocabulary[word] = len(vocabulary)

    return vocabulary


vocab = build_vocabulary(train_df["text"])

print("Vocabulary size:", len(vocab))
print("PAD token index:", vocab[PAD_TOKEN])
print("Unknown token index:", vocab[UNK_TOKEN])

Vocabulary size: 10000
PAD token index: 0
Unknown token index: 1


## Convert emails into numerical sequences

In [19]:
def encode_text(text, vocabulary, max_length=MAX_LENGTH):
    tokens = tokenize(text)

    # Convert each token to its vocabulary index.
    token_ids = [
        vocabulary.get(token, vocabulary[UNK_TOKEN])
        for token in tokens
    ]

    # Truncate long emails.
    token_ids = token_ids[:max_length]

    # Pad short emails.
    if len(token_ids) < max_length:
        token_ids += [vocabulary[PAD_TOKEN]] * (
            max_length - len(token_ids)
        )

    return token_ids


class EmailDataset(Dataset):
    def __init__(self, dataframe, vocabulary):
        self.texts = dataframe["text"].tolist()
        self.labels = dataframe["label"].tolist()
        self.vocabulary = vocabulary

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        input_ids = encode_text(
            self.texts[index],
            self.vocabulary
        )

        return (
            torch.tensor(input_ids, dtype=torch.long),
            torch.tensor(self.labels[index], dtype=torch.long)
        )


train_dataset = EmailDataset(train_df, vocab)
val_dataset = EmailDataset(val_df, vocab)
test_dataset = EmailDataset(test_df, vocab)

print("Training examples:", len(train_dataset))
print("Validation examples:", len(val_dataset))
print("Test examples:", len(test_dataset))

sample_ids, sample_label = train_dataset[0]

print("Example sequence length:", len(sample_ids))
print("Example label:", sample_label.item())

Training examples: 26914
Validation examples: 4751
Test examples: 2000
Example sequence length: 256
Example label: 0


## Define the four models
All four models use the same vocabulary, sequence length, training split, and evaluation data

In [20]:
class BaselineModel(nn.Module):
    """Average word embeddings followed by a classifier."""

    def __init__(self, vocab_size):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            EMBEDDING_DIM,
            padding_idx=0
        )

        self.classifier = nn.Sequential(
            nn.Linear(EMBEDDING_DIM, HIDDEN_DIM),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(HIDDEN_DIM, NUM_CLASSES)
        )

    def forward(self, input_ids):
        embeddings = self.embedding(input_ids)

        mask = (input_ids != 0).unsqueeze(-1).float()

        summed = (embeddings * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1)

        average_embeddings = summed / counts

        return self.classifier(average_embeddings)


class CNNModel(nn.Module):
    """CNN model for detecting local word patterns."""

    def __init__(self, vocab_size):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            EMBEDDING_DIM,
            padding_idx=0
        )

        self.conv = nn.Conv1d(
            EMBEDDING_DIM,
            HIDDEN_DIM,
            kernel_size=3,
            padding=1
        )

        self.pool = nn.AdaptiveMaxPool1d(1)

        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(HIDDEN_DIM, NUM_CLASSES)
        )

    def forward(self, input_ids):
        x = self.embedding(input_ids)
        x = x.transpose(1, 2)

        x = torch.relu(self.conv(x))
        x = self.pool(x).squeeze(-1)

        return self.classifier(x)


class LSTMModel(nn.Module):
    """LSTM model for learning sequence information."""

    def __init__(self, vocab_size):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            EMBEDDING_DIM,
            padding_idx=0
        )

        self.lstm = nn.LSTM(
            input_size=EMBEDDING_DIM,
            hidden_size=HIDDEN_DIM,
            batch_first=True
        )

        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(
            HIDDEN_DIM,
            NUM_CLASSES
        )

    def forward(self, input_ids):
        x = self.embedding(input_ids)

        _, (hidden, _) = self.lstm(x)

        final_hidden = hidden[-1]
        final_hidden = self.dropout(final_hidden)

        return self.classifier(final_hidden)


class TransformerModel(nn.Module):
    """Transformer encoder for learning word relationships."""

    def __init__(self, vocab_size):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            EMBEDDING_DIM,
            padding_idx=0
        )

        self.position_embedding = nn.Embedding(
            MAX_LENGTH,
            EMBEDDING_DIM
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=EMBEDDING_DIM,
            nhead=4,
            dim_feedforward=128,
            dropout=0.2,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=2
        )

        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(EMBEDDING_DIM, NUM_CLASSES)
        )

    def forward(self, input_ids):
        batch_size, sequence_length = input_ids.shape

        positions = torch.arange(
            sequence_length,
            device=input_ids.device
        ).unsqueeze(0).expand(batch_size, -1)

        x = (
            self.embedding(input_ids)
            + self.position_embedding(positions)
        )

        padding_mask = input_ids == 0

        x = self.transformer(
            x,
            src_key_padding_mask=padding_mask
        )

        # Mean pooling over non-padding positions.
        valid_mask = (~padding_mask).unsqueeze(-1).float()

        x = (x * valid_mask).sum(dim=1)
        x = x / valid_mask.sum(dim=1).clamp(min=1)

        return self.classifier(x)


MODEL_CLASSES = {
    "Baseline": BaselineModel,
    "CNN": CNNModel,
    "LSTM": LSTMModel,
    "Transformer": TransformerModel
}

print("Models:", list(MODEL_CLASSES.keys()))

Models: ['Baseline', 'CNN', 'LSTM', 'Transformer']


## Define training and evaluation functions

In [21]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def create_loaders(seed):
    generator = torch.Generator()
    generator.manual_seed(seed)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator,
        num_workers=0
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0
    )

    return train_loader, val_loader, test_loader


def train_one_epoch(model, loader, optimizer, criterion):
    model.train()

    total_loss = 0.0
    total_examples = 0

    for input_ids, labels in loader:
        input_ids = input_ids.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        logits = model(input_ids)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_examples += batch_size

    return total_loss / max(total_examples, 1)


def evaluate_model(model, loader, criterion):
    model.eval()

    total_loss = 0.0
    total_examples = 0

    all_labels = []
    all_predictions = []

    with torch.no_grad():
        for input_ids, labels in loader:
            input_ids = input_ids.to(DEVICE)
            labels = labels.to(DEVICE)

            logits = model(input_ids)
            loss = criterion(logits, labels)

            predictions = torch.argmax(logits, dim=1)

            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size
            total_examples += batch_size

            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(predictions.cpu().numpy())

    average_loss = total_loss / max(total_examples, 1)

    metrics = {
        "loss": average_loss,
        "accuracy": accuracy_score(
            all_labels, all_predictions
        ),
        "precision_macro": precision_score(
            all_labels,
            all_predictions,
            average="macro",
            zero_division=0
        ),
        "recall_macro": recall_score(
            all_labels,
            all_predictions,
            average="macro",
            zero_division=0
        ),
        "macro_f1": f1_score(
            all_labels,
            all_predictions,
            average="macro",
            zero_division=0
        ),
        "spam_f1": f1_score(
            all_labels,
            all_predictions,
            pos_label=1,
            zero_division=0
        )
    }

    return metrics, all_labels, all_predictions

## Train and evaluate all models across three seeds
This cell runs 12 experiments: four models multiplied by three seeds. Each experiment trains for 10 epochs. It selects the best epoch using validation macro F1 and then evaluates the selected model once on the test set.

In [22]:
results = []
training_history = []

for model_name, model_class in MODEL_CLASSES.items():

    for seed in SEEDS:
        print("\n" + "=" * 65)
        print(f"Model: {model_name} | Seed: {seed}")
        print("=" * 65)

        set_seed(seed)

        train_loader, val_loader, test_loader = create_loaders(seed)

        model = model_class(len(vocab)).to(DEVICE)

        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=LEARNING_RATE
        )

        criterion = nn.CrossEntropyLoss()

        best_val_f1 = -1.0
        best_epoch = 0
        best_state = None

        start_time = time.time()

        for epoch in range(1, EPOCHS + 1):
            train_loss = train_one_epoch(
                model,
                train_loader,
                optimizer,
                criterion
            )

            val_metrics, _, _ = evaluate_model(
                model,
                val_loader,
                criterion
            )

            training_history.append({
                "model": model_name,
                "seed": seed,
                "epoch": epoch,
                "train_loss": train_loss,
                "val_loss": val_metrics["loss"],
                "val_accuracy": val_metrics["accuracy"],
                "val_macro_f1": val_metrics["macro_f1"]
            })

            print(
                f"Epoch {epoch:02d}/{EPOCHS} | "
                f"Train Loss: {train_loss:.4f} | "
                f"Val Loss: {val_metrics['loss']:.4f} | "
                f"Val Accuracy: {val_metrics['accuracy']:.4f} | "
                f"Val Macro F1: {val_metrics['macro_f1']:.4f}"
            )

            # Save the best model using validation macro F1.
            if val_metrics["macro_f1"] > best_val_f1:
                best_val_f1 = val_metrics["macro_f1"]
                best_epoch = epoch

                best_state = {
                    key: value.detach().cpu().clone()
                    for key, value in model.state_dict().items()
                }

        # Restore the best validation checkpoint.
        model.load_state_dict(best_state)

        # Test only after model selection is complete.
        test_metrics, y_true, y_pred = evaluate_model(
            model,
            test_loader,
            criterion
        )

        elapsed_seconds = time.time() - start_time

        result = {
            "dataset": DATASET_ID,
            "model": model_name,
            "seed": seed,
            "best_epoch": best_epoch,
            "best_val_macro_f1": best_val_f1,
            "test_loss": test_metrics["loss"],
            "test_accuracy": test_metrics["accuracy"],
            "test_precision_macro": test_metrics["precision_macro"],
            "test_recall_macro": test_metrics["recall_macro"],
            "test_macro_f1": test_metrics["macro_f1"],
            "test_spam_f1": test_metrics["spam_f1"],
            "training_seconds": elapsed_seconds
        }

        results.append(result)

        print("\nTest results:")
        print(json.dumps(result, indent=2))

        # Save each trained model checkpoint.
        model_path = os.path.join(
            OUTPUT_DIR,
            f"{model_name.lower()}_seed_{seed}.pt"
        )

        torch.save(
            {
                "model_name": model_name,
                "seed": seed,
                "model_state_dict": best_state,
                "vocabulary": vocab,
                "max_length": MAX_LENGTH
            },
            model_path
        )

        del model, optimizer

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print("\nAll experiments completed.")


Model: Baseline | Seed: 42
Epoch 01/10 | Train Loss: 0.2525 | Val Loss: 0.0836 | Val Accuracy: 0.9735 | Val Macro F1: 0.9735
Epoch 02/10 | Train Loss: 0.0663 | Val Loss: 0.0525 | Val Accuracy: 0.9836 | Val Macro F1: 0.9836
Epoch 03/10 | Train Loss: 0.0402 | Val Loss: 0.0419 | Val Accuracy: 0.9874 | Val Macro F1: 0.9874
Epoch 04/10 | Train Loss: 0.0254 | Val Loss: 0.0377 | Val Accuracy: 0.9886 | Val Macro F1: 0.9886
Epoch 05/10 | Train Loss: 0.0155 | Val Loss: 0.0374 | Val Accuracy: 0.9884 | Val Macro F1: 0.9884
Epoch 06/10 | Train Loss: 0.0097 | Val Loss: 0.0413 | Val Accuracy: 0.9886 | Val Macro F1: 0.9886
Epoch 07/10 | Train Loss: 0.0060 | Val Loss: 0.0436 | Val Accuracy: 0.9893 | Val Macro F1: 0.9893
Epoch 08/10 | Train Loss: 0.0040 | Val Loss: 0.0449 | Val Accuracy: 0.9893 | Val Macro F1: 0.9893
Epoch 09/10 | Train Loss: 0.0025 | Val Loss: 0.0496 | Val Accuracy: 0.9882 | Val Macro F1: 0.9882
Epoch 10/10 | Train Loss: 0.0020 | Val Loss: 0.0564 | Val Accuracy: 0.9867 | Val Macro F1:

KeyboardInterrupt: 

## Inference Cost Comparison

Measure parameter count, inference time per sample, and peak memory
for the Baseline, CNN, LSTM, and Transformer models across all three seeds.
All models must use the same test inputs and hardware.

In [ ]:

# Inference Cost Comparison

import platform
import time

def count_parameters(model):
    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )


# Use one fixed test batch for every model and seed.
_, _, test_loader = create_loaders(SEEDS[0])
fixed_test_inputs, _ = next(iter(test_loader))
fixed_test_inputs = fixed_test_inputs.to(DEVICE)


def measure_inference_cost(model, test_inputs, repetitions=50):
    model.eval()
    device = next(model.parameters()).device
    inputs = test_inputs.to(device)

    # Warm-up
    with torch.inference_mode():
        for _ in range(5):
            model(inputs)

    if device.type == "cuda":
        torch.cuda.synchronize(device)
        torch.cuda.reset_peak_memory_stats(device)

    start = time.perf_counter()

    with torch.inference_mode():
        for _ in range(repetitions):
            model(inputs)

    if device.type == "cuda":
        torch.cuda.synchronize(device)

    elapsed = time.perf_counter() - start

    ms_per_sample = (
        elapsed * 1000
        / (repetitions * inputs.size(0))
    )

    peak_gpu_memory_mb = None

    if device.type == "cuda":
        peak_gpu_memory_mb = (
            torch.cuda.max_memory_allocated(device)
            / (1024 ** 2)
        )

    return {
        "parameters": count_parameters(model),
        "inference_ms_per_sample": ms_per_sample,
        "peak_gpu_memory_mb": peak_gpu_memory_mb,
        "device": str(device),
    }


cost_results = []

for model_name, model_class in MODEL_CLASSES.items():
    for seed in SEEDS:
        print(f"Measuring {model_name}, seed {seed}...")

        checkpoint_path = os.path.join(
            OUTPUT_DIR,
            f"{model_name.lower()}_seed_{seed}.pt"
        )

        if not os.path.exists(checkpoint_path):
            raise FileNotFoundError(
                f"Missing checkpoint: {checkpoint_path}. "
                "Run the training cell first."
            )

        checkpoint = torch.load(
            checkpoint_path,
            map_location="cpu",
            weights_only=False
        )

        model = model_class(len(vocab))
        model.load_state_dict(checkpoint["model_state_dict"])
        model = model.to(DEVICE)

        cost = measure_inference_cost(
            model,
            fixed_test_inputs,
            repetitions=50
        )

        cost_results.append({
            "model": model_name,
            "seed": seed,
            **cost
        })

        del model

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


cost_df = pd.DataFrame(cost_results)
display(cost_df)

print("Python version:", platform.python_version())
print("PyTorch version:", torch.__version__)
print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CPU:", platform.processor())


Measuring Baseline, seed 42...
Measuring Baseline, seed 123...
Measuring Baseline, seed 2026...
Measuring CNN, seed 42...
Measuring CNN, seed 123...
Measuring CNN, seed 2026...
Measuring LSTM, seed 42...


FileNotFoundError: Checkpoint not found: results/enron_spam_experiment\lstm_seed_42.pt. Run the training cells first.

In [ ]:

# Save inference-cost results

from pathlib import Path

cost_output_dir = Path(OUTPUT_DIR) / "inference_cost"
cost_output_dir.mkdir(parents=True, exist_ok=True)

cost_path = cost_output_dir / "inference_cost_all_seeds.csv"
cost_df.to_csv(cost_path, index=False)

cost_summary_df = cost_df.groupby("model").agg(
    parameters=("parameters", "mean"),
    inference_ms_mean=("inference_ms_per_sample", "mean"),
    inference_ms_std=("inference_ms_per_sample", "std"),
    peak_gpu_memory_mean=("peak_gpu_memory_mb", "mean")
).reset_index()

summary_cost_path = (
    cost_output_dir / "inference_cost_summary.csv"
)
cost_summary_df.to_csv(summary_cost_path, index=False)

display(cost_summary_df)

print("Saved individual measurements:", cost_path)
print("Saved model summary:", summary_cost_path)


## Save the results

In [ ]:
## Save the results

results_df = pd.DataFrame(results)
history_df = pd.DataFrame(training_history)

results_path = os.path.join(
    OUTPUT_DIR,
    "enron_spam_all_results.csv"
)

history_path = os.path.join(
    OUTPUT_DIR,
    "enron_spam_training_history.csv"
)

results_df.to_csv(results_path, index=False)
history_df.to_csv(history_path, index=False)

print("Saved results to:", results_path)
print("Saved training history to:", history_path)

display(results_df)

## Compare average results across seeds

In [ ]:
summary_df = (
    results_df
    .groupby("model")
    .agg(
        mean_accuracy=("test_accuracy", "mean"),
        std_accuracy=("test_accuracy", "std"),
        mean_macro_f1=("test_macro_f1", "mean"),
        std_macro_f1=("test_macro_f1", "std"),
        mean_spam_f1=("test_spam_f1", "mean"),
        std_spam_f1=("test_spam_f1", "std")
    )
    .reset_index()
    .sort_values("mean_macro_f1", ascending=False)
)

summary_path = os.path.join(
    OUTPUT_DIR,
    "enron_spam_model_comparison.csv"
)

summary_df.to_csv(summary_path, index=False)

display(summary_df)

print("Saved comparison to:", summary_path)

## Show the best model's detailed classification report

In [ ]:
best_result = results_df.sort_values(
    "best_val_macro_f1",
    ascending=False
).iloc[0]

best_model_name = best_result["model"]
best_seed = int(best_result["seed"])

print("Best individual run:")
print(best_result)

# Rebuild and reload that model.
best_model = MODEL_CLASSES[best_model_name](
    len(vocab)
).to(DEVICE)

checkpoint_path = os.path.join(
    OUTPUT_DIR,
    f"{best_model_name.lower()}_seed_{best_seed}.pt"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

best_model.load_state_dict(
    checkpoint["model_state_dict"]
)

_, _, test_loader = create_loaders(best_seed)

criterion = nn.CrossEntropyLoss()

best_metrics, y_true, y_pred = evaluate_model(
    best_model,
    test_loader,
    criterion
)

print("\nClassification Report:")
print(
    classification_report(
        y_true,
        y_pred,
        labels=[0, 1],
        target_names=["Ham", "Spam"],
        zero_division=0
    )
)

print("Confusion Matrix:")
print(
    confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    )
)

## Save experiment metadata

In [ ]:
metadata = {
    "group": "MVPs",
    "course": "Advanced Deep Learning - AIGC-5500-0NA",
    "dataset_id": DATASET_ID,
    "train_rows": len(train_df),
    "validation_rows": len(val_df),
    "test_rows": len(test_df),
    "vocabulary_size": len(vocab),
    "max_length": MAX_LENGTH,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "seeds": SEEDS,
    "models": list(MODEL_CLASSES.keys()),
    "device": str(DEVICE),
    "label_mapping": {
        "0": "ham",
        "1": "spam"
    }
}

metadata_path = os.path.join(
    OUTPUT_DIR,
    "experiment_metadata.json"
)

with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=4)

print("Metadata saved to:", metadata_path)
print("\nExperiment files:")
for filename in sorted(os.listdir(OUTPUT_DIR)):
    print("-", filename)